# Evolution-1 playground
Evolve a four-legged walker, compare runs, and watch the best one. Edit only the SETTINGS cell, then run all cells (Runtime > Run all on Colab).

## 1. Install and load
On Colab this installs the project from GitHub. Locally it just uses the repo folder.

In [ ]:
import os, subprocess, sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "evolution-simulation[research] @ git+https://github.com/000wahab000/Evolution"], check=True)
else:  # running from <repo>/notebooks or <repo>
    root = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
    sys.path.insert(0, root)
    os.chdir(root)  # results/ always lands in the repo root

import json
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np

from src.research.playground import make_cfg, show_walker, train

## 2. Settings
The only cell you need to change.

In [ ]:
CPG = True            # True: the brain gets a rhythm clock as input. False: no clock (cpg_mode "zeros").
MUTATION_RATE = 0.15  # chance each of the 324 weights is changed when breeding (0.05 = gentle, 0.4 = wild)
SEEDS = [0, 1]        # one independent evolution per seed (run one after another); more = more trustworthy
GENERATIONS = 100     # rounds of breeding per seed. Measured 1.6 s/gen locally (2 workers, 50 walkers), so
                      # 2 seeds take ~5 min here and roughly 8-10 min on Colab's 2 CPUs
POPULATION = 50       # walkers per generation
RUN_NAME = "my_run"   # name of this run; the result is saved to results/playground/<RUN_NAME>.json
START_FROM = "random"  # "random", or "pretrained" to start around the repo's saved best walker
KEEP_TRAINING = False  # True: continue from the saved checkpoint and add GENERATIONS more
CHECKPOINT_EVERY = 10  # save progress every N generations (results/playground/<RUN_NAME>_seed<s>.ckpt)
CHART_EVERY = 5        # redraw the best/mean distance chart every N generations (0 = off)
PREVIEW_EVERY = 0      # show a short animation of the current best walker every N generations (0 = off)

## 3. Run the experiment
Each seed evolves walkers with the repo's genetic algorithm (one line per generation, plus a live chart), then tests the best one on fresh terrains it never trained on. Set `KEEP_TRAINING = True` and run again to train longer from the last checkpoint.

In [ ]:
results = []
with ProcessPoolExecutor(min(POPULATION, os.cpu_count() or 1)) as pool:  # only walker scoring is parallel
    for s in SEEDS:
        results.append(train(s, CPG, MUTATION_RATE, GENERATIONS, POPULATION, RUN_NAME, pool=pool,
                             keep_training=KEEP_TRAINING, checkpoint_every=CHECKPOINT_EVERY,
                             start_from=START_FROM, chart_every=CHART_EVERY, preview_every=PREVIEW_EVERY))
best = [r[0] for r in results]
champion = results[int(np.argmax(best))][1]
settings = dict(CPG=CPG, MUTATION_RATE=MUTATION_RATE, SEEDS=SEEDS, GENERATIONS=GENERATIONS,
                POPULATION=POPULATION, RUN_NAME=RUN_NAME, START_FROM=START_FROM)
os.makedirs("results/playground", exist_ok=True)
with open(f"results/playground/{RUN_NAME}.json", "w") as f:
    json.dump({"settings": settings, "best_distance_per_seed": dict(zip(map(str, SEEDS), best)),
               "champion_genome": champion}, f)
print("best distance per seed:", dict(zip(SEEDS, np.round(best, 1))))

## 4. Compare runs
Each bar is the average best distance (test terrains) of one saved run; dots are the individual seeds. Your current run is orange.

In [ ]:
runs = []
for name in sorted(os.listdir("results/playground")):
    if name.endswith(".json"):
        with open(f"results/playground/{name}") as f:
            runs.append((name[:-5], list(json.load(f)["best_distance_per_seed"].values())))
fig, ax = plt.subplots(figsize=(1.5 + 1.2 * len(runs), 4))
for i, (name, d) in enumerate(runs):
    ax.bar(i, np.mean(d), color="tab:orange" if name == RUN_NAME else "tab:blue")
    ax.scatter([i] * len(d), d, color="black", zorder=3, s=15)
ax.set_xticks(range(len(runs)), [n for n, _ in runs], rotation=30, ha="right")
ax.set_ylabel("distance walked")
plt.tight_layout()
plt.show()

## 5. Watch the best walker
Replays the best genome of this run on one test terrain and animates it (side view; the brown line is the ground).

In [ ]:
show_walker(champion, make_cfg(CPG, SEEDS[0]))